# Sentiment Analysis of Internship Feedback
This Jupyter notebook builds an end-to-end NLP sentiment classification pipeline using **TF-IDF + Logistic Regression** (baseline) and **Hugging Face Transformers** (DistilBERT / pipeline) to analyze intern reviews, classify sentiment, and extract areas for improving intern satisfaction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print('Libraries imported successfully!')

## Step 1: Load and Clean the Feedback Dataset (`intern_feedback_data.csv`)

In [ ]:
# Load dataset
df = pd.read_csv('intern_feedback_data.csv')

print('--- Dataset Info ---')
print(df.info())

# Data Cleaning: Drop missing rows and duplicates
df = df.dropna(subset=['feedback_text', 'sentiment'])
df = df.drop_duplicates()

print(f'Cleaned Dataset Shape: {df.shape}')
df.head(10)

## Step 2: Exploratory Data Analysis (Sentiment Distribution)

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='sentiment', palette='viridis', order=['Positive', 'Neutral', 'Negative'])
plt.title('Distribution of Intern Feedback Sentiments', fontsize=12, fontweight='bold')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

## Step 3: Text Vectorization & Train-Test Split (Logistic Regression)

In [ ]:
X = df['feedback_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f'Training features shape: {X_train_tfidf.shape}')

## Step 4: Model Training & Evaluation (Logistic Regression)

In [ ]:
lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred = lr_model.predict(X_test_tfidf)

print('--- Classification Report ---')
print(classification_report(y_test, y_pred))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

## Step 5: Advanced Transformer Sentiment Pipeline (Hugging Face)

In [ ]:
from transformers import pipeline

# Load pre-trained sentiment analysis transformer pipeline
print('Loading Hugging Face sentiment pipeline...')
sentiment_pipeline = pipeline('sentiment-analysis')

# Test on a few sample reviews
sample_reviews = df['feedback_text'].head(5).tolist()
results = sentiment_pipeline(sample_reviews)

for review, res in zip(sample_reviews, results):
    print(f'Review: "{review}"')
    print(f'Prediction: {res["label"]} (Confidence: {res["score"]:.4f})\n')

## Step 6: Actionable Insights & Improvement Areas

In [ ]:
# Extract keywords / analyze negative feedback to pinpoint satisfaction issues
negative_df = df[df['sentiment'] == 'Negative']
print(f'Total Negative Feedback Count: {len(negative_df)}')
print('Sample Negative Feedback (Areas to Improve):')
for text in negative_df['feedback_text'].head(5):
    print('-', text)